In [14]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, variation, levene
from statsmodels.stats.multitest import multipletests
import seaborn as sns
import matplotlib.pyplot as plt

In [15]:
path_load = 'D:/bioTest/data/'
path_save = 'D:/bioTest/data/'
df = pd.read_excel(f"{path_load}data_yak.xlsx", index_col=0)
feats_slctd = pd.read_excel(f"{path_load}feats_selected.xlsx", index_col=0).index.values

In [16]:
df_stat = pd.DataFrame(index=list(feats_slctd))
for feat in list(feats_slctd):
    vals = {}
    for group in ['Central', 'Yakutia']:
        vals[group] = df.loc[df['Region'] == group, feat].values
        df_stat.at[feat, f"mean_{group}"] = np.mean(vals[group])
        df_stat.at[feat, f"median_{group}"] = np.median(vals[group])
        df_stat.at[feat, f"q75_{group}"], df_stat.at[feat, f"q25_{group}"] = np.percentile(vals[group], [75 , 25])
        df_stat.at[feat, f"iqr_{group}"] = df_stat.at[feat, f"q75_{group}"] - df_stat.at[feat, f"q25_{group}"]
        df_stat.at[feat, f"variation_{group}"] = variation(vals[group])
    _, df_stat.at[feat, "mw_pval"] = mannwhitneyu(vals['Central'], vals['Yakutia'], alternative='two-sided')
    _, df_stat.at[feat, "levene_pval"] = levene(vals['Central'], vals['Yakutia'], center='mean')
_, df_stat.loc[feats_slctd, "mw_pval_fdr_bh"], _, _ = multipletests(df_stat.loc[feats_slctd, "mw_pval"].values, 0.05, method='fdr_bh')
_, df_stat.loc[feats_slctd, "mw_pval_bonferroni"], _, _ = multipletests(df_stat.loc[feats_slctd, "mw_pval"].values, 0.05, method='bonferroni')
_, df_stat.loc[feats_slctd, "mw_pval_simes-hochberg"], _, _ = multipletests(df_stat.loc[feats_slctd, "mw_pval"].values, 0.05, method='simes-hochberg')
_, df_stat.loc[feats_slctd, "levene_pval_fdr_bh"], _, _ = multipletests(df_stat.loc[feats_slctd, "levene_pval"].values, 0.05, method='fdr_bh')
_, df_stat.loc[feats_slctd, "levene_pval_bonferroni"], _, _ = multipletests(df_stat.loc[feats_slctd, "levene_pval"].values, 0.05, method='bonferroni')
_, df_stat.loc[feats_slctd, "levene_pval_simes-hochberg"], _, _ = multipletests(df_stat.loc[feats_slctd, "levene_pval"].values, 0.05, method='simes-hochberg')
df_stat.sort_values([f"mw_pval_fdr_bh"], ascending=[True], inplace=True)
df_stat.to_excel(f"{path_save}stat.xlsx", index_label='Features')

In [18]:
df_fig = df_stat.loc[feats_slctd, :]
df_fig.sort_values([f"mw_pval_fdr_bh"], ascending=[True], inplace=True)
df_fig['Features'] = df_fig.index
df_fig['levene_pval_fdr_bh_log'] = -np.log10(df_fig['levene_pval_fdr_bh'])
df_fig['color'] = 'skyblue'
df_fig.loc[df_fig['levene_pval_fdr_bh'] < 0.05, 'color'] = 'blue'

sns.set_theme(style='ticks')
fig, ax = plt.subplots(figsize=(3, 15))
barplot = sns.barplot(
    data=df_fig,
    y='Features',
    x='levene_pval_fdr_bh_log',
    edgecolor='black',
    palette=df_fig['color'].values,
    width=0.4,
    #hue='Features',
    #dodge=False,
    ax=ax,
)
ax.set_xlabel(r"$-\log_{10}(\mathrm{p-value})$", fontsize=18)
# ax.get_legend().remove()
ax.set(xlim=(0, 20))
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
ax.set_ylabel('', fontsize=20)
ax.set_xticklabels([f"{int(tick):d}" for tick in ax.get_xticks()], fontsize=16)
ax.set_yticklabels(ax.get_yticklabels(), fontsize = 16)
plt.savefig(f"{path_save}barplot_levene.png", bbox_inches='tight', dpi=200)
plt.savefig(f"{path_save}barplot_levene.pdf", bbox_inches='tight')
plt.close(fig)

C:\Users\alena\AppData\Local\Temp\ipykernel_38736\16590344.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  barplot = sns.barplot(
C:\Users\alena\AppData\Local\Temp\ipykernel_38736\16590344.py:10: UserWarning: Numpy array is not a supported type for `palette`. Please convert your palette to a list. This will become an error in v0.14
  barplot = sns.barplot(
C:\Users\alena\AppData\Local\Temp\ipykernel_38736\16590344.py:27: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{int(tick):d}" for tick in ax.get_xticks()], fontsize=16)
C:\Users\alena\AppData\Local\Temp\ipykernel_38736\16590344.py:28: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(ax.get

In [ ]:
df_fig = df_stat.loc[feats_slctd, :]
df_fig.sort_values([f"mw_pval_fdr_bh"], ascending=[True], inplace=True)
df_fig['Features'] = df_fig.index
df_fig['mw_pval_fdr_bh_log'] = -np.log10(df_fig['mw_pval_fdr_bh'])
df_fig['levene_pval_fdr_bh_log'] = -np.log10(df_fig['levene_pval_fdr_bh'])
df_melt = df_fig.melt(value_vars=['mw_pval_fdr_bh_log', 'levene_pval_fdr_bh_log'], var_name='Test', value_name='p-value', ignore_index=False)
df_melt.loc[(df_melt['Test'] == 'mw_pval_fdr_bh_log') & (df_melt['p-value'] > -np.log10(0.05)), 'Test groups'] = 'Mann-Whitney\nSignificant'
df_melt.loc[(df_melt['Test'] == 'mw_pval_fdr_bh_log') & (df_melt['p-value'] <= -np.log10(0.05)), 'Test groups'] = 'Mann-Whitney\nNon-significant'
df_melt.loc[(df_melt['Test'] == 'levene_pval_fdr_bh_log') & (df_melt['p-value'] > -np.log10(0.05)), 'Test groups'] = 'Levene\nSignificant'
df_melt.loc[(df_melt['Test'] == 'levene_pval_fdr_bh_log') & (df_melt['p-value'] <= -np.log10(0.05)), 'Test groups'] = 'Levene\nNon-significant'
df_melt['Features'] = df_melt.index

palette = {
    'Mann-Whitney\nSignificant': 'red',
    'Mann-Whitney\nNon-significant': 'pink',
    'Levene\nSignificant': 'blue',
    'Levene\nNon-significant': 'skyblue',
}

sns.set_theme(style='ticks')
fig, ax = plt.subplots(figsize=(3, 15))
barplot = sns.barplot(
    data=df_melt,
    y='Features',
    x='p-value',
    hue='Test groups',
    edgecolor='black',
    palette=palette,
    #dodge=False,
    ax=ax,
)

ax.set_xlabel(r"$-\log_{10}(\mathrm{p-value})$", fontsize=18)
# ax.get_legend().remove()
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
ax.set_ylabel('', fontsize=20)
ax.set_xticklabels([f"{int(tick):d}" for tick in ax.get_xticks()], fontsize=16)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=16)
plt.savefig(f"{path_save}barplot_new.png", bbox_inches='tight', dpi=200)
plt.savefig(f"{path_save}barplot_new.pdf", bbox_inches='tight')
plt.close(fig)

C:\Users\alena\AppData\Local\Temp\ipykernel_38736\372461299.py:38: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{int(tick):d}" for tick in ax.get_xticks()], fontsize=16)
C:\Users\alena\AppData\Local\Temp\ipykernel_38736\372461299.py:39: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels(ax.get_yticklabels(), fontsize=16)
